In [165]:
import numpy as np
import pandas as pd

from collections import deque


pd.set_option('display.max_columns', None)

In [166]:
df = pd.read_csv("../data/raw/1993-94.csv")
df.head()

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27
0,E0,14/08/93,Arsenal,Coventry,0.0,3.0,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E0,14/08/93,Aston Villa,QPR,4.0,1.0,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E0,14/08/93,Chelsea,Blackburn,1.0,2.0,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E0,14/08/93,Liverpool,Sheffield Weds,2.0,0.0,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E0,14/08/93,Man City,Leeds,1.0,1.0,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [167]:
df.shape

(552, 28)

In [168]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 552 entries, 0 to 551
Data columns (total 28 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Div          462 non-null    str    
 1   Date         462 non-null    str    
 2   HomeTeam     462 non-null    str    
 3   AwayTeam     462 non-null    str    
 4   FTHG         462 non-null    float64
 5   FTAG         462 non-null    float64
 6   FTR          462 non-null    str    
 7   Unnamed: 7   0 non-null      float64
 8   Unnamed: 8   0 non-null      float64
 9   Unnamed: 9   0 non-null      float64
 10  Unnamed: 10  0 non-null      float64
 11  Unnamed: 11  0 non-null      float64
 12  Unnamed: 12  0 non-null      float64
 13  Unnamed: 13  0 non-null      float64
 14  Unnamed: 14  0 non-null      float64
 15  Unnamed: 15  0 non-null      float64
 16  Unnamed: 16  0 non-null      float64
 17  Unnamed: 17  0 non-null      float64
 18  Unnamed: 18  0 non-null      float64
 19  Unnamed: 19  0 non-

In [169]:
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 552 entries, 0 to 551
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Div       462 non-null    str    
 1   Date      462 non-null    str    
 2   HomeTeam  462 non-null    str    
 3   AwayTeam  462 non-null    str    
 4   FTHG      462 non-null    float64
 5   FTAG      462 non-null    float64
 6   FTR       462 non-null    str    
dtypes: float64(2), str(5)
memory usage: 30.3 KB


In [170]:
df = df.drop_duplicates()

In [171]:
df = df.dropna(how='all')

In [172]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 462 entries, 0 to 461
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Div       462 non-null    str    
 1   Date      462 non-null    str    
 2   HomeTeam  462 non-null    str    
 3   AwayTeam  462 non-null    str    
 4   FTHG      462 non-null    float64
 5   FTAG      462 non-null    float64
 6   FTR       462 non-null    str    
dtypes: float64(2), str(5)
memory usage: 25.4 KB


In [173]:
df.tail()

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
457,E0,07/05/94,Sheffield Weds,Man City,1.0,1.0,D
458,E0,07/05/94,Swindon,Leeds,0.0,5.0,A
459,E0,07/05/94,Tottenham,QPR,1.0,2.0,A
460,E0,07/05/94,West Ham,Southampton,3.0,3.0,D
461,E0,08/05/94,Man United,Coventry,0.0,0.0,D


In [174]:
# # ========== 1. INICIALIZAR ESTRUCTURAS DE DATOS ==========
# # Usamos defaultdict para evitar tener que usar setdefault todo el tiempo
# from collections import defaultdict

# # Historial de resultados (G/E/P)
# historial_resultados = defaultdict(list)      # Todos los partidos
# historial_resultados_local = defaultdict(list)  # Solo como local
# historial_resultados_visitante = defaultdict(list)  # Solo como visitante

# # Historial de goles (marcados, encajados)
# historial_goles = defaultdict(list)           # Todos los partidos
# historial_goles_local = defaultdict(list)     # Solo como local
# historial_goles_visitante = defaultdict(list) # Solo como visitante

# # Historial de enfrentamientos directos (por par de equipos)
# historial_h2h = defaultdict(list)             # Todos los enfrentamientos
# historial_h2h_local = defaultdict(list)       # Enfrentamientos donde el equipo es LOCAL

# # Para almacenar estadísticas acumuladas de la temporada
# goles_temporada = defaultdict(lambda: {'gf': 0, 'gc': 0, 'partidos': 0})

# # Listas para almacenar resultados de cada fila
# results = {
#     'racha_general_home': [], 'racha_general_away': [],
#     'racha_local_home': [], 'racha_visitante_away': [],
#     'goles_prom_general_home_marcados': [], 'goles_prom_general_home_encajados': [],
#     'goles_prom_general_away_marcados': [], 'goles_prom_general_away_encajados': [],
#     'goles_prom_local_home_marcados': [], 'goles_prom_local_home_encajados': [],
#     'goles_prom_visitante_away_marcados': [], 'goles_prom_visitante_away_encajados': [],
#     'dif_goles_general_home': [], 'dif_goles_general_away': [],
#     'dif_goles_local_home': [], 'dif_goles_visitante_away': [],
#     'goles_total_temporada_home_marcados': [], 'goles_total_temporada_home_encajados': [],
#     'goles_total_temporada_away_marcados': [], 'goles_total_temporada_away_encajados': [],
#     'racha_h2h_home': [], 'racha_h2h_away': [],
#     'racha_h2h_local_home': [], 'racha_h2h_visitante_away': []
# }

# # ========== 2. FUNCIÓN AUXILIAR PARA CALCULAR MÉTRICAS ==========
# def calcular_metricas(lista_resultados, lista_goles, ventana=5):
#     """
#     Calcula racha (string) y estadísticas de goles a partir de listas
#     Retorna: (racha_str, goles_marcados_prom, goles_encajados_prom, diferencia)
#     """
#     if not lista_resultados:
#         return '', 0.0, 0.0, 0
    
#     # Tomar últimos 'ventana' elementos
#     resultados = lista_resultados[-ventana:]
#     goles = lista_goles[-ventana:] if lista_goles else []
    
#     # Calcular racha como string
#     racha = ''.join(resultados)
    
#     # Calcular goles
#     if goles:
#         gf = sum(g[0] for g in goles)
#         gc = sum(g[1] for g in goles)
#         n = len(goles)
#         prom_gf = gf / n
#         prom_gc = gc / n
#         dif = gf - gc
#     else:
#         prom_gf = prom_gc = 0.0
#         dif = 0
    
#     return racha, prom_gf, prom_gc, dif

# # ========== 3. RECORRER PARTIDOS ==========
# for idx, row in df.iterrows():
#     home = row['HomeTeam']
#     away = row['AwayTeam']
#     g_home = row['FTHG']
#     g_away = row['FTAG']
#     resultado = row['FTR']
    
#     # ========== A) CALCULAR MÉTRICAS PARA EL EQUIPO LOCAL ==========
#     # General
#     racha_gen_h, gf_gen_h, gc_gen_h, dif_gen_h = calcular_metricas(
#         historial_resultados[home], historial_goles[home]
#     )
#     # Como local
#     racha_loc_h, gf_loc_h, gc_loc_h, dif_loc_h = calcular_metricas(
#         historial_resultados_local[home], historial_goles_local[home]
#     )
    
#     # ========== B) CALCULAR MÉTRICAS PARA EL EQUIPO VISITANTE ==========
#     # General
#     racha_gen_a, gf_gen_a, gc_gen_a, dif_gen_a = calcular_metricas(
#         historial_resultados[away], historial_goles[away]
#     )
#     # Como visitante
#     racha_vis_a, gf_vis_a, gc_vis_a, dif_vis_a = calcular_metricas(
#         historial_resultados_visitante[away], historial_goles_visitante[away]
#     )
    
#     # ========== C) ESTADÍSTICAS DE TEMPORADA (TOTAL ACUMULADO) ==========
#     # Antes del partido, obtener acumulados
#     temp_home_gf = goles_temporada[home]['gf']
#     temp_home_gc = goles_temporada[home]['gc']
#     temp_away_gf = goles_temporada[away]['gf']
#     temp_away_gc = goles_temporada[away]['gc']
    
#     # ========== D) ENFRENTAMIENTOS DIRECTOS (H2H) ==========
#     # Clave única para el par de equipos (orden alfabético para consistencia)
#     key_h2h = tuple(sorted([home, away]))
    
#     # Racha H2H para el LOCAL (resultados del local contra este visitante)
#     h2h_local_list = historial_h2h[key_h2h]
#     # Filtrar partidos donde el LOCAL es el equipo que nos interesa
#     h2h_local_resultados = [r for eq, r in h2h_local_list if eq == home]
#     racha_h2h_home = ''.join(h2h_local_resultados[-5:]) if h2h_local_resultados else ''
    
#     # Racha H2H para el VISITANTE (resultados del visitante contra este local)
#     h2h_visitante_resultados = [r for eq, r in h2h_local_list if eq == away]
#     racha_h2h_away = ''.join(h2h_visitante_resultados[-5:]) if h2h_visitante_resultados else ''
    
#     # Racha H2H como LOCAL (partidos donde el equipo juega en casa contra este rival)
#     h2h_local_casa_list = historial_h2h_local[key_h2h]
#     h2h_local_casa_resultados = [r for eq, r in h2h_local_casa_list if eq == home]
#     racha_h2h_local_home = ''.join(h2h_local_casa_resultados[-5:]) if h2h_local_casa_resultados else ''
    
#     # Racha H2H como VISITANTE (partidos donde el equipo juega fuera contra este rival)
#     h2h_visitante_fuera_list = historial_h2h_local[key_h2h]  # Reutilizamos, pero filtramos por visitante
#     h2h_visitante_fuera_resultados = [r for eq, r in h2h_visitante_fuera_list if eq == away]
#     racha_h2h_visitante_away = ''.join(h2h_visitante_fuera_resultados[-5:]) if h2h_visitante_fuera_resultados else ''
    
#     # ========== E) GUARDAR RESULTADOS EN LISTAS ==========
#     results['racha_general_home'].append(racha_gen_h)
#     results['racha_general_away'].append(racha_gen_a)
#     results['racha_local_home'].append(racha_loc_h)
#     results['racha_visitante_away'].append(racha_vis_a)
    
#     results['goles_prom_general_home_marcados'].append(gf_gen_h)
#     results['goles_prom_general_home_encajados'].append(gc_gen_h)
#     results['goles_prom_general_away_marcados'].append(gf_gen_a)
#     results['goles_prom_general_away_encajados'].append(gc_gen_a)
    
#     results['goles_prom_local_home_marcados'].append(gf_loc_h)
#     results['goles_prom_local_home_encajados'].append(gc_loc_h)
#     results['goles_prom_visitante_away_marcados'].append(gf_vis_a)
#     results['goles_prom_visitante_away_encajados'].append(gc_vis_a)
    
#     results['dif_goles_general_home'].append(dif_gen_h)
#     results['dif_goles_general_away'].append(dif_gen_a)
#     results['dif_goles_local_home'].append(dif_loc_h)
#     results['dif_goles_visitante_away'].append(dif_vis_a)
    
#     results['goles_total_temporada_home_marcados'].append(temp_home_gf)
#     results['goles_total_temporada_home_encajados'].append(temp_home_gc)
#     results['goles_total_temporada_away_marcados'].append(temp_away_gf)
#     results['goles_total_temporada_away_encajados'].append(temp_away_gc)
    
#     results['racha_h2h_home'].append(racha_h2h_home)
#     results['racha_h2h_away'].append(racha_h2h_away)
#     results['racha_h2h_local_home'].append(racha_h2h_local_home)
#     results['racha_h2h_visitante_away'].append(racha_h2h_visitante_away)
    
#     # ========== F) ACTUALIZAR HISTORIALES CON EL PARTIDO ACTUAL ==========
#     # Determinar resultados G/E/P
#     if resultado == 'H':
#         res_local, res_visitante = 'G', 'P'
#     elif resultado == 'A':
#         res_local, res_visitante = 'P', 'G'
#     else:
#         res_local, res_visitante = 'E', 'E'
    
#     # Actualizar historiales de resultados
#     historial_resultados[home].append(res_local)
#     historial_resultados[away].append(res_visitante)
#     historial_resultados_local[home].append(res_local)
#     historial_resultados_visitante[away].append(res_visitante)
    
#     # Actualizar historiales de goles
#     historial_goles[home].append((g_home, g_away))
#     historial_goles[away].append((g_away, g_home))
#     historial_goles_local[home].append((g_home, g_away))
#     historial_goles_visitante[away].append((g_away, g_home))
    
#     # Actualizar estadísticas de temporada
#     goles_temporada[home]['gf'] += g_home
#     goles_temporada[home]['gc'] += g_away
#     goles_temporada[home]['partidos'] += 1
#     goles_temporada[away]['gf'] += g_away
#     goles_temporada[away]['gc'] += g_home
#     goles_temporada[away]['partidos'] += 1
    
#     # Actualizar H2H
#     # Para el LOCAL en este partido
#     historial_h2h[key_h2h].append((home, res_local))
#     # Para el VISITANTE en este partido
#     historial_h2h[key_h2h].append((away, res_visitante))
#     # H2H como local (solo para quien juega en casa)
#     historial_h2h_local[key_h2h].append((home, res_local))

# # ========== 4. AÑADIR TODAS LAS COLUMNAS AL DATAFRAME ==========
# for col_name, col_data in results.items():
#     df[col_name] = col_data

# # ========== 5. MOSTRAR RESULTADOS ==========
# print("Columnas añadidas:")
# print([col for col in results.keys()])
# print("\n" + "="*100 + "\n")

# # Mostrar primeras filas con columnas seleccionadas
# columnas_mostrar = ['HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
#                     'racha_general_home', 'racha_general_away',
#                     'goles_prom_general_home_marcados', 'goles_prom_general_home_encajados',
#                     'dif_goles_general_home',
#                     'goles_total_temporada_home_marcados', 'goles_total_temporada_home_encajados',
#                     'racha_h2h_home', 'racha_h2h_away']


In [ ]:
# # ================= 1. IMPORTS =================
# from collections import defaultdict, deque

# # ================= 2. INICIALIZAR ESTRUCTURAS =================

# # Historial de resultados y goles (solo últimos 5)
# ventana = 5
# historial_resultados = defaultdict(lambda: deque(maxlen=ventana))
# historial_resultados_local = defaultdict(lambda: deque(maxlen=ventana))
# historial_resultados_visitante = defaultdict(lambda: deque(maxlen=ventana))

# historial_goles = defaultdict(lambda: deque(maxlen=ventana))
# historial_goles_local = defaultdict(lambda: deque(maxlen=ventana))
# historial_goles_visitante = defaultdict(lambda: deque(maxlen=ventana))

# # Historial H2H optimizado
# historial_h2h = defaultdict(lambda: {'home': deque(maxlen=ventana),
#                                     'away': deque(maxlen=ventana)})
# historial_h2h_local = defaultdict(lambda: deque(maxlen=ventana))  # local vs visitante

# # Acumulados de temporada
# goles_temporada = defaultdict(lambda: {'gf': 0, 'gc': 0, 'partidos': 0})

# # Resultados finales
# results = {
#     'racha_general_home': [], 'racha_general_away': [],
#     'racha_local_home': [], 'racha_visitante_away': [],
#     'goles_prom_general_home_marcados': [], 'goles_prom_general_home_encajados': [],
#     'goles_prom_general_away_marcados': [], 'goles_prom_general_away_encajados': [],
#     'goles_prom_local_home_marcados': [], 'goles_prom_local_home_encajados': [],
#     'goles_prom_visitante_away_marcados': [], 'goles_prom_visitante_away_encajados': [],
#     'dif_goles_general_home': [], 'dif_goles_general_away': [],
#     'dif_goles_local_home': [], 'dif_goles_visitante_away': [],
#     'goles_total_temporada_home_marcados': [], 'goles_total_temporada_home_encajados': [],
#     'goles_total_temporada_away_marcados': [], 'goles_total_temporada_away_encajados': [],
#     'racha_h2h_home': [], 'racha_h2h_away': [],
#     'racha_h2h_local_home': [], 'racha_h2h_visitante_away': []
# }

# # ================= 3. FUNCIONES AUXILIARES =================
# def calcular_metricas(resultados_deque, goles_deque):
#     """
#     Calcula racha (string) y estadísticas de goles a partir de deques
#     Retorna: racha_str, goles_marcados_prom, goles_encajados_prom, diferencia
#     """
#     if not resultados_deque:
#         return '', 0.0, 0.0, 0
    
#     # Racha como string
#     racha = ''.join(resultados_deque)
    
#     # Goles
#     if goles_deque:
#         gf = sum(g[0] for g in goles_deque)
#         gc = sum(g[1] for g in goles_deque)
#         n = len(goles_deque)
#         return racha, gf/n, gc/n, gf-gc
#     else:
#         return racha, 0.0, 0.0, 0

# def resultados_por_equipo(resultado, home_team, away_team):
#     """
#     Devuelve resultados G/E/P para local y visitante
#     """
#     if resultado == 'H':
#         return 'G', 'P'
#     elif resultado == 'A':
#         return 'P', 'G'
#     else:
#         return 'E', 'E'

# # ================= 4. RECORRER DATAFRAME =================
# for row in df.itertuples(index=False):
#     home = row.HomeTeam
#     away = row.AwayTeam
#     g_home = row.FTHG
#     g_away = row.FTAG
#     resultado = row.FTR

#     # ---- METRICAS EQUIPO LOCAL ----
#     racha_gen_h, gf_gen_h, gc_gen_h, dif_gen_h = calcular_metricas(
#         historial_resultados[home], historial_goles[home]
#     )
#     racha_loc_h, gf_loc_h, gc_loc_h, dif_loc_h = calcular_metricas(
#         historial_resultados_local[home], historial_goles_local[home]
#     )

#     # ---- METRICAS EQUIPO VISITANTE ----
#     racha_gen_a, gf_gen_a, gc_gen_a, dif_gen_a = calcular_metricas(
#         historial_resultados[away], historial_goles[away]
#     )
#     racha_vis_a, gf_vis_a, gc_vis_a, dif_vis_a = calcular_metricas(
#         historial_resultados_visitante[away], historial_goles_visitante[away]
#     )

#     # ---- ESTADISTICAS TEMPORADA ----
#     temp_home_gf = goles_temporada[home]['gf']
#     temp_home_gc = goles_temporada[home]['gc']
#     temp_away_gf = goles_temporada[away]['gf']
#     temp_away_gc = goles_temporada[away]['gc']

#     # ---- H2H ----
#     key_h2h = tuple(sorted([home, away]))
#     racha_h2h_home = ''.join(historial_h2h[key_h2h]['home'])
#     racha_h2h_away = ''.join(historial_h2h[key_h2h]['away'])
#     racha_h2h_local_home = ''.join([r for r in historial_h2h_local[key_h2h] if r == 'G' or r == 'E' or r == 'P'])  # local deque
#     racha_h2h_visitante_away = ''.join([r for r in historial_h2h_local[key_h2h] if r == 'G' or r == 'E' or r == 'P'])  # visitante deque

#     # ---- GUARDAR RESULTADOS ----
#     results['racha_general_home'].append(racha_gen_h)
#     results['racha_general_away'].append(racha_gen_a)
#     results['racha_local_home'].append(racha_loc_h)
#     results['racha_visitante_away'].append(racha_vis_a)

#     results['goles_prom_general_home_marcados'].append(gf_gen_h)
#     results['goles_prom_general_home_encajados'].append(gc_gen_h)
#     results['goles_prom_general_away_marcados'].append(gf_gen_a)
#     results['goles_prom_general_away_encajados'].append(gc_gen_a)

#     results['goles_prom_local_home_marcados'].append(gf_loc_h)
#     results['goles_prom_local_home_encajados'].append(gc_loc_h)
#     results['goles_prom_visitante_away_marcados'].append(gf_vis_a)
#     results['goles_prom_visitante_away_encajados'].append(gc_vis_a)

#     results['dif_goles_general_home'].append(dif_gen_h)
#     results['dif_goles_general_away'].append(dif_gen_a)
#     results['dif_goles_local_home'].append(dif_loc_h)
#     results['dif_goles_visitante_away'].append(dif_vis_a)

#     results['goles_total_temporada_home_marcados'].append(temp_home_gf)
#     results['goles_total_temporada_home_encajados'].append(temp_home_gc)
#     results['goles_total_temporada_away_marcados'].append(temp_away_gf)
#     results['goles_total_temporada_away_encajados'].append(temp_away_gc)

#     results['racha_h2h_home'].append(racha_h2h_home)
#     results['racha_h2h_away'].append(racha_h2h_away)
#     results['racha_h2h_local_home'].append(racha_h2h_local_home)
#     results['racha_h2h_visitante_away'].append(racha_h2h_visitante_away)

#     # ---- ACTUALIZAR HISTORIALES ----
#     res_local, res_visitante = resultados_por_equipo(resultado, home, away)

#     # Resultados
#     historial_resultados[home].append(res_local)
#     historial_resultados[away].append(res_visitante)
#     historial_resultados_local[home].append(res_local)
#     historial_resultados_visitante[away].append(res_visitante)

#     # Goles
#     historial_goles[home].append((g_home, g_away))
#     historial_goles[away].append((g_away, g_home))
#     historial_goles_local[home].append((g_home, g_away))
#     historial_goles_visitante[away].append((g_away, g_home))

#     # Temporada
#     goles_temporada[home]['gf'] += g_home
#     goles_temporada[home]['gc'] += g_away
#     goles_temporada[home]['partidos'] += 1
#     goles_temporada[away]['gf'] += g_away
#     goles_temporada[away]['gc'] += g_home
#     goles_temporada[away]['partidos'] += 1

#     # H2H
#     historial_h2h[key_h2h]['home'].append(res_local)
#     historial_h2h[key_h2h]['away'].append(res_visitante)
#     historial_h2h_local[key_h2h].append(res_local)

# # ================= 5. AÑADIR COLUMNAS =================
# for col, data in results.items():
#     df[col] = data


In [176]:
df[(df["HomeTeam"] == "Arsenal") | (df["AwayTeam"] == "Arsenal")]

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,racha_general_home,racha_general_away,racha_local_home,racha_visitante_away,goles_prom_general_home_marcados,goles_prom_general_home_encajados,goles_prom_general_away_marcados,goles_prom_general_away_encajados,goles_prom_local_home_marcados,goles_prom_local_home_encajados,goles_prom_visitante_away_marcados,goles_prom_visitante_away_encajados,dif_goles_general_home,dif_goles_general_away,dif_goles_local_home,dif_goles_visitante_away,goles_total_temporada_home_marcados,goles_total_temporada_home_encajados,goles_total_temporada_away_marcados,goles_total_temporada_away_encajados,racha_h2h_home,racha_h2h_away,racha_h2h_local_home,racha_h2h_visitante_away
0,E0,14/08/93,Arsenal,Coventry,0.0,3.0,A,,,,,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,,,,
11,E0,16/08/93,Tottenham,Arsenal,0.0,1.0,A,G,P,,,1.000000,0.0,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,1.0,-3.0,0.0,0.0,1.0,0.0,0.0,3.0,,,,
29,E0,21/08/93,Sheffield Weds,Arsenal,0.0,1.0,A,PE,PG,E,G,0.000000,1.0,0.500000,1.500000,0.000000,0.000000,1.000000,0.000000,-2.0,-2.0,0.0,1.0,0.0,2.0,1.0,3.0,,,,
34,E0,24/08/93,Arsenal,Leeds,2.0,1.0,H,PGG,EGP,P,E,0.666667,1.0,0.666667,1.666667,0.000000,3.000000,1.000000,1.000000,-1.0,-3.0,-3.0,0.0,2.0,3.0,2.0,5.0,,,,
45,E0,28/08/93,Arsenal,Everton,2.0,0.0,H,PGGG,GGGP,PG,GP,1.000000,1.0,1.750000,0.750000,1.000000,2.000000,1.000000,0.500000,0.0,4.0,-2.0,1.0,4.0,4.0,7.0,3.0,,,,
59,E0,01/09/93,Blackburn,Arsenal,1.0,1.0,D,GPGGE,PGGGG,PG,GG,1.600000,1.0,1.200000,0.800000,1.500000,1.500000,1.000000,0.000000,3.0,2.0,0.0,2.0,8.0,5.0,6.0,4.0,,,,
66,E0,11/09/93,Arsenal,Ipswich,4.0,0.0,H,GGGGE,GGPEE,PGG,GPE,1.400000,0.4,0.800000,0.600000,1.333333,1.333333,1.333333,0.666667,5.0,1.0,0.0,2.0,7.0,5.0,7.0,3.0,,,,
86,E0,19/09/93,Man United,Arsenal,1.0,0.0,H,EGGGP,GGGEG,GEG,GGE,1.800000,0.8,2.000000,0.400000,2.333333,0.333333,1.000000,0.333333,5.0,8.0,6.0,2.0,14.0,4.0,11.0,5.0,,,,
88,E0,25/09/93,Arsenal,Southampton,1.0,0.0,H,GGEGP,GPPPP,PGGG,PPPP,1.800000,0.6,1.200000,1.800000,2.000000,1.000000,0.250000,1.500000,6.0,-3.0,4.0,-5.0,11.0,6.0,7.0,14.0,,,,
101,E0,02/10/93,Liverpool,Arsenal,0.0,0.0,D,GPPPP,GEGPG,GPGP,GGEP,0.400000,1.0,1.600000,0.400000,1.250000,0.750000,0.750000,0.500000,-3.0,6.0,2.0,1.0,13.0,8.0,12.0,6.0,,,,
